# Proyecto Integrador · Comercio exterior de Argentina y Uruguay
## Notebook 1: carga y limpieza de datos (ETL)

**Fuente:** World Bank – WITS (*World Integrated Trade Solution*), descargas de *Country Profile* con todos los socios (Partner = ALL).

**Objetivo de este notebook:** tomar los 8 archivos Excel descargados de WITS, limpiarlos y dejarlos
en el formato que necesita la base de datos MySQL.

| Entrada (8 archivos) | Salida (2 archivos) |
|---|---|
| Exportaciones e importaciones en US$ y en % de participación, para Argentina y Uruguay | `dim_socio.csv` y `fact_comercio.csv` |

**Flujo del proyecto:** Excel (WITS) → **Python (este notebook): limpieza → CSV** → MySQL (`ddl.sql` + `carga.sql`) → Power BI

### Cómo usarlo en Google Colab (no necesita acceso a Google Drive)
1. Abrir [Google Colab](https://colab.research.google.com) → *Archivo → Subir notebook* → elegir este archivo.
2. Menú **Entorno de ejecución → Ejecutar todas**.
3. En el **Paso 1** aparece un botón **"Elegir archivos"**: seleccionar los 8 Excel de WITS de una sola vez
   (con Ctrl + clic). El notebook sigue solo.
4. Al final se descargan los archivos generados.

> Si Colab muestra el cartel *"¿Permitir que este cuaderno acceda a tus archivos de Google Drive?"*, responder
> **No**. Este notebook no usa Drive.

### Alternativa sin cuenta de Google: JupyterLite
Abrir [jupyter.org/try-jupyter/lab](https://jupyter.org/try-jupyter/lab), arrastrar este notebook y los 8 Excel
al panel de la izquierda y elegir *Run → Run All Cells*. Para bajar los resultados: clic derecho sobre el archivo → *Download*.

## Paso 0 · Librerías
Usamos solo **pandas**, la librería estándar de Python para trabajar con tablas, **glob**, que sirve para
buscar archivos por nombre, y **os**, para saber en qué carpeta estamos.

In [1]:
# Instalar el lector de archivos Excel (en Colab ya viene instalado; en JupyterLite hace falta)
%pip install -q openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import glob
import pandas as pd

# Mostrar los números con 2 decimales y sin notación científica
pd.set_option("display.float_format", "{:,.2f}".format)

## Paso 1 · Encontrar los archivos
Buscamos **todos los archivos Excel (`.xlsx`)**, sin importar cómo se llamen,
y nos quedamos con los que tienen el formato de WITS (una hoja con la columna `Reporter Name`).
Deberían ser 8: 2 países × 2 flujos (exportación / importación) × 2 medidas (US$ / % de participación).

**¿En qué carpeta busca?** Si el notebook está dentro del repositorio del proyecto (carpeta `notebooks/`),
lee los Excel de `data/raw/` y guarda los CSV en `data/processed/`. Si no, usa la misma carpeta del notebook.
Si algo falla, la celda muestra la **carpeta donde está buscando**: los Excel tienen que estar ahí.

In [3]:
# Si estamos en Google Colab y todavía no están los Excel, mostrar el botón "Elegir archivos".
# Los archivos se suben directo desde la computadora, sin pasar por Google Drive.
try:
    from google.colab import files
    if len(glob.glob("*.xlsx")) < 8:
        print("Seleccioná los 8 archivos Excel de WITS (podés elegirlos todos juntos con Ctrl + clic):")
        files.upload()
except ImportError:
    pass  # No estamos en Colab: los archivos ya tienen que estar en la misma carpeta que el notebook

In [4]:
import os

# Estructura del repositorio: notebooks/ está al lado de data/raw/ y data/processed/.
# Si esas carpetas existen se usan; si no, todo se lee y se guarda en la carpeta del notebook.
CARPETA_ENTRADA = "../data/raw" if os.path.isdir("../data/raw") else "."
CARPETA_SALIDA = "../data/processed" if os.path.isdir("../data/processed") else "."


def es_archivo_wits(ruta):
    """Devuelve True si el Excel tiene alguna hoja con la columna 'Reporter Name' (formato WITS)."""
    try:
        hojas = pd.read_excel(ruta, sheet_name=None, nrows=1)
        return any("Reporter Name" in t.columns for t in hojas.values())
    except Exception:
        return False   # archivo dañado, abierto en Excel o que no es una planilla


print("Carpeta donde se buscan los archivos:", os.path.abspath(CARPETA_ENTRADA), "\n")

todos_los_excel = sorted(glob.glob(os.path.join(CARPETA_ENTRADA, "*.xlsx")))
archivos = [a for a in todos_los_excel if es_archivo_wits(a)]
descartados = [a for a in todos_los_excel if a not in archivos]

print(f"Se encontraron {len(archivos)} archivos de WITS:")
for a in archivos:
    print("  -", a)
if descartados:
    print("\nExcel ignorados (no tienen formato WITS):", descartados)

if len(archivos) == 0:
    # Detener el notebook acá: sin archivos no tiene sentido seguir
    raise FileNotFoundError(
        "No hay archivos de WITS en la carpeta " + os.path.abspath(CARPETA_ENTRADA) +
        ". Copiar ahí los 8 Excel y volver a ejecutar todo."
    )
elif len(archivos) != 8:
    print("\n⚠️ ATENCIÓN: se esperaban 8 archivos. Si hay más, puede haber archivos viejos o repetidos")
    print("   (por ejemplo, 'nombre (1).xlsx'): dejar en la carpeta solo los 8 que se quieren usar.")

Carpeta donde se buscan los archivos: .../comercio-argentina-uruguay/data/raw 

Se encontraron 8 archivos de WITS:
  - ../data/raw/WITS_ARG_export_participacion.xlsx
  - ../data/raw/WITS_ARG_export_usd.xlsx
  - ../data/raw/WITS_ARG_import_participacion.xlsx
  - ../data/raw/WITS_ARG_import_usd.xlsx
  - ../data/raw/WITS_URY_export_participacion.xlsx
  - ../data/raw/WITS_URY_export_usd.xlsx
  - ../data/raw/WITS_URY_import_participacion.xlsx
  - ../data/raw/WITS_URY_import_usd.xlsx


## Paso 2 · Leer y limpiar cada archivo

Todos los archivos de WITS tienen la misma estructura, así que escribimos **una sola función** y la
aplicamos a los 8. La función hace lo siguiente:

1. Busca la hoja que tiene los datos. WITS siempre agrega una `Sheet1` vacía, y la hoja con datos cambia de nombre
   según la descarga (`Partner-Timeseries` o `Product-TimeSeries-Partner`). Por eso la buscamos por su contenido:
   es la que tiene la columna `Reporter Name`.
2. **Elimina filas sin país**. Por ejemplo, en el archivo de importaciones de Argentina hay una fila suelta
   al final con un solo número.
3. **Quita espacios sobrantes** en los textos (WITS trae `" World"` con un espacio al inicio).
4. **Pasa de formato ancho a formato largo**: en lugar de una columna por año, queda una fila por
   país–socio–flujo–año. Este formato es el que necesitan MySQL y Power BI.
5. **Elimina las celdas vacías**, porque un año sin dato no aporta información.
6. Detecta si el archivo trae **US$** o **% de participación**, leyendo la columna `Indicator`.

```
FORMATO ANCHO (original)                    FORMATO LARGO (resultado)
País      Socio   2019   2020   ...         País      Socio   Año   Valor
Argentina Brasil  10385  7941               Argentina Brasil  2019  10385
                                            Argentina Brasil  2020  7941
```

In [5]:
def leer_archivo(ruta):
    """Lee un archivo de WITS y lo devuelve limpio y en formato largo."""

    # 1. Leer todas las hojas y quedarnos con la que tiene la columna "Reporter Name"
    hojas = pd.read_excel(ruta, sheet_name=None)   # devuelve {nombre_hoja: tabla}
    df = next(t for t in hojas.values() if "Reporter Name" in t.columns)

    # 2. Eliminar filas sin país informante (filas sueltas o vacías)
    df = df.dropna(subset=["Reporter Name"])

    # 3. Quitar espacios al inicio y al final de las columnas de texto
    for col in ["Reporter Name", "Partner Name", "Trade Flow", "Indicator"]:
        df[col] = df[col].str.strip()

    # 6. ¿Este archivo es de US$ o de % de participación?
    indicador = df["Indicator"].iloc[0]
    if "US$" in indicador:
        medida = "valor_miles_usd"
    else:
        medida = "participacion_pct"

    # 4. Las columnas de años son las que tienen solo dígitos ("2019", "2020", ...)
    columnas_anios = [c for c in df.columns if str(c).isdigit()]

    # 4. Pasar de ancho a largo (en Power Query esto se llama "Anular dinamización de columnas")
    largo = df.melt(
        id_vars=["Reporter Name", "Partner Name", "Trade Flow"],
        value_vars=columnas_anios,
        var_name="anio",
        value_name=medida,
    )

    # 5. Eliminar los años sin dato
    largo = largo.dropna(subset=[medida])

    # Renombrar columnas al español y dejar el año como número entero
    largo = largo.rename(columns={
        "Reporter Name": "pais_reportante",
        "Partner Name": "socio",
        "Trade Flow": "flujo",
    })
    largo["anio"] = largo["anio"].astype(int)

    print(f"{ruta:42} → {medida:18} {len(largo):6,} filas  "
          f"({largo['anio'].min()}–{largo['anio'].max()})")
    return largo, medida

In [6]:
# Aplicar la función a los 8 archivos y separar los resultados según la medida
tablas_usd = []
tablas_pct = []

for ruta in archivos:
    tabla, medida = leer_archivo(ruta)
    if medida == "valor_miles_usd":
        tablas_usd.append(tabla)
    else:
        tablas_pct.append(tabla)

# Unir (apilar) los archivos de cada medida en una sola tabla
usd = pd.concat(tablas_usd, ignore_index=True)
pct = pd.concat(tablas_pct, ignore_index=True)

print(f"\nTotal filas en US$: {len(usd):,}")
print(f"Total filas en %:   {len(pct):,}")
usd.head()

../data/raw/WITS_ARG_export_participacion.xlsx → participacion_pct   5,485 filas  (1993–2023)
../data/raw/WITS_ARG_export_usd.xlsx       → valor_miles_usd     5,485 filas  (1993–2023)
../data/raw/WITS_ARG_import_participacion.xlsx → participacion_pct   4,948 filas  (1994–2023)
../data/raw/WITS_ARG_import_usd.xlsx       → valor_miles_usd     4,932 filas  (1994–2023)


../data/raw/WITS_URY_export_participacion.xlsx → participacion_pct   4,627 filas  (1994–2023)
../data/raw/WITS_URY_export_usd.xlsx       → valor_miles_usd     4,627 filas  (1994–2023)
../data/raw/WITS_URY_import_participacion.xlsx → participacion_pct   4,491 filas  (1994–2023)
../data/raw/WITS_URY_import_usd.xlsx       → valor_miles_usd     4,479 filas  (1994–2023)

Total filas en US$: 19,523
Total filas en %:   19,551


,pais_reportante,socio,flujo,anio,valor_miles_usd
0,Argentina,World,Export,1993,"13,117,745.15"
1,Argentina,Afghanistan,Export,1993,248.30
2,Argentina,Algeria,Export,1993,"8,312.93"
3,Argentina,Andorra,Export,1993,551.48
4,Argentina,Angola,Export,1993,"6,954.77"


## Paso 3 · Juntar US$ y % en una sola tabla
Unimos las dos tablas por **país + socio + flujo + año**. Usamos una unión **"outer"** (completa),
para no perder filas que tengan una medida y no la otra. Por ejemplo, para Argentina hay % de
exportación desde 1993, pero US$ solo desde 2019.

In [7]:
claves = ["pais_reportante", "socio", "flujo", "anio"]

# Control: no debe haber filas repetidas antes de unir
assert not usd.duplicated(claves).any(), "Hay filas duplicadas en US$"
assert not pct.duplicated(claves).any(), "Hay filas duplicadas en %"

comercio = usd.merge(pct, on=claves, how="outer")
print(f"Filas después de unir: {len(comercio):,}")
comercio.head()

Filas después de unir: 19,551


,pais_reportante,socio,flujo,anio,valor_miles_usd,participacion_pct
0,Argentina,Afghanistan,Export,1993,248.30,0.00
1,Argentina,Afghanistan,Export,1998,23.79,0.00
2,Argentina,Afghanistan,Export,2002,84.33,0.00
3,Argentina,Afghanistan,Export,2003,165.00,0.00
4,Argentina,Afghanistan,Export,2004,450.98,0.00


## Paso 4 · Clasificar a los socios comerciales
La columna de socios **mezcla países con totales y regiones**. Si se suman todos, el comercio se cuenta
dos veces: por ejemplo, Brasil ya está incluido dentro de *Latin America & Caribbean* y dentro de *World*.
Por eso clasificamos cada socio en un `tipo_socio`:

| tipo_socio | Qué incluye | ¿Se suma con otros? |
|---|---|---|
| `Total` | *World*, el total del país con el mundo | No, es el total |
| `Region` | Las 7 regiones del Banco Mundial | No, se superponen con los países |
| `Otro` | *Unspecified*, *Free Zones*, *Special Categories* (destino no identificado) | Sí, son parte del total |
| `Pais` | Todo lo demás | Sí |

**Regla para el análisis:** para rankings o sumas de socios, filtrar `tipo_socio = 'Pais'`.
Para el total de un país, usar `tipo_socio = 'Total'`.

*Nota:* "Other Asia, nes" es el nombre que usa la ONU para **Taiwán**, así que se considera país.

In [8]:
REGIONES = [
    "East Asia & Pacific",
    "Europe & Central Asia",
    "Latin America & Caribbean",
    "Middle East, North Africa, Afghanistan & Pakistan",
    "North America",
    "South Asia",
    "Sub-Saharan Africa",
]
OTROS = ["Unspecified", "Free Zones", "Special Categories"]


def clasificar_socio(nombre):
    if nombre == "World":
        return "Total"
    if nombre in REGIONES:
        return "Region"
    if nombre in OTROS:
        return "Otro"
    return "Pais"


comercio["tipo_socio"] = comercio["socio"].apply(clasificar_socio)

# Cuántos socios distintos hay de cada tipo
comercio.groupby("tipo_socio")["socio"].nunique()

tipo_socio
Otro        3
Pais      236
Region      7
Total       1
Name: socio, dtype: int64

## Paso 5 · Controles de calidad
Antes de cargar los datos verificamos que tengan sentido. Estos controles se pueden contar en el
informe técnico como parte de la validación.

### 5.1 ¿Qué años hay para cada país y flujo?

In [9]:
# Primer y último año con dato, por separado para US$ y para %
anios_usd = usd.groupby(["pais_reportante", "flujo"])["anio"].agg(["min", "max"])
anios_pct = pct.groupby(["pais_reportante", "flujo"])["anio"].agg(["min", "max"])

cobertura = pd.concat({"US$": anios_usd, "%": anios_pct}, axis=1)
cobertura

US$           %      
                         min   max   min   max
pais_reportante flujo                         
Argentina       Export  1993  2023  1993  2023
                Import  1994  2023  1994  2023
Uruguay         Export  1994  2023  1994  2023
                Import  1994  2023  1994  2023

👉 **Lectura:** la tabla muestra el primer y el último año con dato de cada archivo. Si el rango no coincide
entre los dos países, la comparación debe limitarse a los años en común. En la descarga actual ambos países
cubren **1994–2023**, de modo que se pueden comparar 30 años completos.

### 5.2 ¿La suma de los socios coincide con el total *World*?
Sumamos los socios de tipo `Pais` + `Otro` y lo comparamos con la fila *World*.
La diferencia debería ser chica, de menos del 1 %. Una diferencia pequeña es normal: WITS no publica
por separado algunos socios muy chicos o confidenciales, pero sí los incluye en el total *World*.

In [10]:
con_usd = comercio.dropna(subset=["valor_miles_usd"])

suma_socios = (
    con_usd[con_usd["tipo_socio"].isin(["Pais", "Otro"])]
    .groupby(["pais_reportante", "flujo", "anio"])["valor_miles_usd"].sum()
)
total_world = (
    con_usd[con_usd["tipo_socio"] == "Total"]
    .groupby(["pais_reportante", "flujo", "anio"])["valor_miles_usd"].sum()
)

control = pd.DataFrame({"suma_socios": suma_socios, "world": total_world})
control["diferencia_pct"] = (control["suma_socios"] / control["world"] - 1) * 100

print(f"Diferencia máxima: {control['diferencia_pct'].abs().max():.4f} %")
control.tail(10)

Diferencia máxima: 0.4206 %


suma_socios         world  diferencia_pct
pais_reportante flujo  anio                                            
Uruguay         Import 2014 10,733,896.55 10,762,296.79           -0.26
                       2015  9,456,570.15  9,489,416.54           -0.35
                       2016  8,112,095.00  8,136,629.14           -0.30
                       2017  8,443,362.36  8,457,876.67           -0.17
                       2018  8,871,751.18  8,893,246.00           -0.24
                       2019  8,219,492.98  8,245,758.80           -0.32
                       2020  7,532,672.12  7,564,488.84           -0.42
                       2021 10,292,760.39 10,320,437.77           -0.27
                       2022 12,939,740.08 12,973,460.92           -0.26
                       2023 12,454,083.39 12,486,195.28           -0.26

### 5.3 ¿El % de participación coincide con US$ socio ÷ US$ total?
Recalculamos el porcentaje con los montos y lo comparamos con el que informa WITS.

In [11]:
world = (
    con_usd[con_usd["tipo_socio"] == "Total"]
    [["pais_reportante", "flujo", "anio", "valor_miles_usd"]]
    .rename(columns={"valor_miles_usd": "total_world"})
)
chequeo = comercio.merge(world, on=["pais_reportante", "flujo", "anio"])
chequeo = chequeo.dropna(subset=["valor_miles_usd", "participacion_pct"])
chequeo["pct_calculado"] = chequeo["valor_miles_usd"] / chequeo["total_world"] * 100
chequeo["diferencia"] = (chequeo["pct_calculado"] - chequeo["participacion_pct"]).abs()

print(f"Filas comparadas: {len(chequeo):,}")
print(f"Diferencia máxima: {chequeo['diferencia'].max():.4f} puntos porcentuales")
print("(WITS redondea el % a 4 decimales, así que diferencias muy chicas son normales)")

Filas comparadas: 19,523
Diferencia máxima: 0.0000 puntos porcentuales
(WITS redondea el % a 4 decimales, así que diferencias muy chicas son normales)


## Paso 6 · Armar las tablas finales del modelo
Separamos los datos en dos tablas, siguiendo un **modelo estrella**:

- **`dim_socio`** (dimensión): un registro por socio comercial, con su id y su tipo.
- **`fact_comercio`** (hechos): los valores numéricos. En lugar del nombre del socio guarda su `id_socio`.

Así el nombre del socio se escribe una sola vez y la tabla de hechos queda más liviana.

In [12]:
# --- Dimensión de socios ---
dim_socio = (
    comercio[["socio", "tipo_socio"]]
    .drop_duplicates()
    .sort_values("socio")
    .reset_index(drop=True)
    .rename(columns={"socio": "nombre_socio"})
)
dim_socio.insert(0, "id_socio", range(1, len(dim_socio) + 1))

# --- Tabla de hechos ---
fact_comercio = (
    comercio.merge(dim_socio, left_on="socio", right_on="nombre_socio")
    [["pais_reportante", "id_socio", "flujo", "anio", "valor_miles_usd", "participacion_pct"]]
    .sort_values(["pais_reportante", "flujo", "anio", "id_socio"])
    .reset_index(drop=True)
)

# Redondear: US$ a 3 decimales y % a 4 decimales, igual que en el ddl.sql
fact_comercio["valor_miles_usd"] = fact_comercio["valor_miles_usd"].round(3)
fact_comercio["participacion_pct"] = fact_comercio["participacion_pct"].round(4)

print(f"dim_socio:     {len(dim_socio):,} filas")
print(f"fact_comercio: {len(fact_comercio):,} filas")
fact_comercio.head()

dim_socio:     247 filas
fact_comercio: 19,551 filas


,pais_reportante,id_socio,flujo,anio,valor_miles_usd,participacion_pct
0,Argentina,1,Export,1993,248.30,0.00
1,Argentina,3,Export,1993,"8,312.93",0.06
2,Argentina,5,Export,1993,551.48,0.00
3,Argentina,6,Export,1993,"6,954.77",0.05
4,Argentina,8,Export,1993,67.82,0.00


## Paso 7 · Exportar a CSV
Guardamos las dos tablas como archivos **CSV** (texto separado por comas), el formato estándar para
pasar datos de una herramienta a otra. Después MySQL los lee con el script `carga.sql`.

Detalles para que MySQL los lea sin problemas:
- **UTF-8**, para que se conserven los acentos y caracteres especiales (por ejemplo *Curaçao*).
- **Fin de línea `\n`**, igual en Windows y en Mac.
- **Celdas vacías** donde no hay dato. En la carga se convierten en `NULL`.

In [13]:
dim_socio.to_csv(os.path.join(CARPETA_SALIDA, "dim_socio.csv"), index=False, encoding="utf-8", lineterminator="\n")
fact_comercio.to_csv(os.path.join(CARPETA_SALIDA, "fact_comercio.csv"), index=False, encoding="utf-8", lineterminator="\n")

print("✅ Archivos generados en", os.path.abspath(CARPETA_SALIDA), ": dim_socio.csv, fact_comercio.csv")

✅ Archivos generados en .../comercio-argentina-uruguay/data/processed : dim_socio.csv, fact_comercio.csv


In [14]:
# Descargar los archivos (solo funciona en Google Colab; en otro entorno quedan en la carpeta)
try:
    from google.colab import files
    for nombre in ["dim_socio.csv", "fact_comercio.csv"]:
        files.download(os.path.join(CARPETA_SALIDA, nombre))
except ImportError:
    print("No estás en Colab: los archivos quedaron guardados en", os.path.abspath(CARPETA_SALIDA))

No estás en Colab: los archivos quedaron guardados en .../comercio-argentina-uruguay/data/processed


## Próximos pasos (en MySQL Workbench)
1. Ejecutar **`ddl.sql`**, que crea la base y las tablas vacías.
2. Ejecutar **`carga.sql`**, que lee los dos CSV y llena las tablas.
3. Verificar con `SELECT COUNT(*) FROM fact_comercio;`. Debe dar la misma cantidad de filas que muestra el Paso 6.
4. Continuar con las consultas de `analisis.sql` y luego conectar Power BI.